# Libs

In [67]:
import warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from astropy.visualization import ZScaleInterval
from astropy.io import fits
from astropy.wcs import WCS
from astropy.wcs.utils import proj_plane_pixel_scales
from astropy.utils.exceptions import AstropyWarning
from pathlib import Path
from typing import Tuple, Dict, Optional, Union
from matplotlib.lines import Line2D
import matplotlib.patches as patches
from scipy.ndimage import distance_transform_edt
import cv2
from scipy.ndimage import distance_transform_edt
from astropy.visualization import ZScaleInterval, AsinhStretch, ImageNormalize

# Plot params

In [70]:
# ====================================================================
# Publication-quality plot settings (ApJ, MNRAS, A&A standards)
# ====================================================================
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman", "Times New Roman"],
    "axes.labelsize": 14,
    "axes.titlesize": 14,
    "legend.fontsize": 12,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
    "axes.linewidth": 1.2,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "savefig.bbox": "tight"
})

# Objects params

In [73]:
# Ваш словарь объектов
obj_dist_step_pc = {
    "Auriga": [450.0, 4, "Lada et al. 2009"],
    "L1495": [140.0, 2, "Galli et al. 2018 / Schlafly et al. 2014"],
    "L43_Karoly_2023": [125.0, 2, "Ortiz-León et al. 2017"],
    "MonR2": [830.0, 5, "Dzib et al. 2016"],
    "NGC1333": [299.0, 5, "Zucker et al. 2018"],
    "OMC1_450": [388.0, 5, "Kounkel et al. 2017"],
    "OMC1_850": [388.0, 5, "Kounkel et al. 2017"],
    "Perseus_B1": [293.0, 3, "Zucker et al. 2018"],
    "Rosette": [1330.0, 6, "Heyer et al. 2006"],
    "l1689b": [147.0, 6, "Ortiz-León et al. 2017"],
    "oph_a": [139.0, 6, "Zucker et al. 2019"],
    "oph_b": [139.0, 6, "Zucker et al. 2019"],
    "oph_c": [139.0, 6, "Zucker et al. 2019"]
}

# Funcs

In [76]:
class PolarizationAnalyzer:
    """
    Pipeline for analyzing dust polarization maps. 
    Focuses on generating structural maps and the polarization angle 
    structure function without assuming an underlying turbulence model.
    """
    def __init__(self, base_archive_path: Union[str, Path]):
        """
        Initializes the analyzer with the root directory of the archive.
        
        Args:
            base_archive_path (str or Path): Path to the BISTRO data archive.
            
        Raises:
            FileNotFoundError: If the provided directory does not exist.
        """
        self.base_dir = Path(base_archive_path)
        if not self.base_dir.exists():
            raise FileNotFoundError(f"Base directory not accessible: {self.base_dir}")
        
    def _load_fits(self, file_path: Path) -> Tuple[np.ndarray, Optional[WCS]]:
        """
        Loads FITS data and collapses degenerate axes.
        
        Args:
            file_path (Path): Path to the target FITS file.
            
        Returns:
            Tuple[np.ndarray, Optional[WCS]]: 2D image array and its celestial WCS.
            
        Raises:
            FileNotFoundError: If the FITS file is missing.
        """
        if not file_path.exists():
            raise FileNotFoundError(f"File not found: {file_path}")
            
        with fits.open(file_path) as hdul:
            data = np.squeeze(hdul[0].data)
            header = hdul[0].header
            
            with warnings.catch_warnings():
                warnings.simplefilter('ignore', AstropyWarning)
                wcs = WCS(header) if 'CTYPE1' in header else None
                if wcs is not None and wcs.pixel_n_dim > 2:
                    wcs = wcs.celestial
        return data, wcs
    
    def _get_robust_limits(self, I: np.ndarray) -> Tuple[float, float]:
        """
        Safely computes display limits, falling back to percentiles if ZScale fails.
        """
        valid_I = I[np.isfinite(I)]
        if valid_I.size == 0:
            return 0.0, 1.0 # Fallback for completely empty maps
            
        try:
            interval = ZScaleInterval()
            vmin, vmax = interval.get_limits(valid_I)
            # Ensure vmax is strictly greater than vmin
            if vmax <= vmin:
                raise ValueError("ZScale produced invalid limits.")
        except Exception:
            # Fallback to robust percentiles
            vmin, vmax = np.percentile(valid_I, [2, 98])
            
        # Absolute final safety check
        if vmax <= vmin:
            vmin = np.nanmin(valid_I)
            vmax = np.nanmax(valid_I)
            if vmax <= vmin: # Map is perfectly flat
                 vmax = vmin + 1.0 
                 
        return vmin, vmax
    
    def load_stokes_maps(self, object_name: str, band: str = "850um") -> Tuple[np.ndarray, np.ndarray, np.ndarray, WCS]:
        """
        Loads Stokes I, Q, and U maps for a specific target.
        
        The method first attempts to find files using the strict BISTRO naming convention:
        '{object_name}_{band}_{stokes}_map.fits'. 
        If strict files are not found, it falls back to a flexible search for any 
        FITS file containing '{stokes}_map' in its filename, accommodating both 
        '.fits' and '.fit' extensions.

        Args:
            object_name (str): Target directory name.
            band (str): Observation band (e.g., "850um").
            
        Returns:
            Tuple[np.ndarray, np.ndarray, np.ndarray, WCS]: Synchronized I, Q, U maps and WCS.
            
        Raises:
            FileNotFoundError: If any of the Stokes components cannot be located.
        """
        maps_dir = self.base_dir / object_name / "maps"
        if not maps_dir.exists():
            raise FileNotFoundError(f"Maps directory not found: {maps_dir}")

        stokes_data = {}
        for component in ['I', 'Q', 'U']:
            # 1. Primary strict search
            strict_name = f"{object_name}_{band}_{component}_map.fits"
            file_path = maps_dir / strict_name
            
            # 2. Fallback flexible search (now accepting .fit and .fits)
            if not file_path.exists():
                flexible_pattern = f"*{component}_map*.fit*"
                found_files = list(maps_dir.glob(flexible_pattern))
                
                if found_files:
                    file_path = found_files[0]  # Take the first matching file
                else:
                    raise FileNotFoundError(
                        f"Could not find Stokes {component} map for {object_name}. "
                        f"Tried strict '{strict_name}' and pattern '{flexible_pattern}'."
                    )
            
            data, wcs_loaded = self._load_fits(file_path)
            stokes_data[component] = data
            
            # Use WCS from the Stokes I map as the reference
            if component == 'I':
                wcs = wcs_loaded

        I, Q, U = stokes_data['I'], stokes_data['Q'], stokes_data['U']

        # Shape synchronization check to prevent broadcasting errors
        if Q.shape != I.shape or U.shape != I.shape:
            print(f"[Warning] Shape mismatch for {object_name}. Synchronizing Q/U to I: {I.shape}")
            ny, nx = I.shape
            Q = Q[:ny, :nx]
            U = U[:ny, :nx]
            
        return I, Q, U, wcs

    @staticmethod
    def compute_evpa(Q: np.ndarray, U: np.ndarray) -> np.ndarray:
        """
        Computes the Electric Vector Position Angle (EVPA).
        
        Args:
            Q (np.ndarray): Stokes Q map.
            U (np.ndarray): Stokes U map.
            
        Returns:
            np.ndarray: EVPA map in degrees (-90 to +90).
        """
        return np.degrees(0.5 * np.arctan2(U, Q))

    @staticmethod
    def pol_vector_components(p: Union[float, np.ndarray], psi_rad: Union[float, np.ndarray]) -> Tuple[np.ndarray, np.ndarray]:
        """
        Calculates Cartesian vector components for plotting the B-field.
        Includes a +pi/2 rotation to convert EVPA to magnetic field orientation.
        
        Args:
            p (float or np.ndarray): Polarization degree or constant length.
            psi_rad (float or np.ndarray): EVPA in radians.
            
        Returns:
            Tuple[np.ndarray, np.ndarray]: X and Y vector components (vx, vy).
        """
        psi_shifted = psi_rad + np.pi / 2.0
        return p * np.cos(psi_shifted), p * np.sin(psi_shifted)

    @staticmethod
    def estimate_rms(data: np.ndarray) -> float:
        """
        Robustly estimates the Root Mean Square (RMS) noise using the Median Absolute Deviation (MAD).
        
        Args:
            data (np.ndarray): 2D map array (e.g., Stokes I or polarized intensity).
            
        Returns:
            float: Estimated background noise level. Returns 0.0 if map is empty/invalid.
        """
        valid = data[np.isfinite(data) & (data != 0)]
        if valid.size == 0: return 0.0
        return 1.4826 * np.median(np.abs(valid - np.median(valid)))

    def get_physical_scales(self, wcs: WCS, distance_pc: float, shape: Tuple[int, int]) -> Dict[str, float]:
        """
        Converts pixel grid scales to physical dimensions.
        
        Args:
            wcs (WCS): World Coordinate System object.
            distance_pc (float): Distance to the source in parsecs.
            shape (Tuple[int, int]): Dimensions of the image (Ny, Nx).
            
        Returns:
            Dict[str, float]: Dictionary containing pixel scales (arcsec, pc) and total field of view.
        """
        pix_scale_deg = np.mean(proj_plane_pixel_scales(wcs))
        pix_scale_arcsec = pix_scale_deg * 3600.0
        pix_scale_pc = np.radians(pix_scale_deg) * distance_pc
        
        return {
            "distance_pc": distance_pc,
            "pix_scale_arcsec": pix_scale_arcsec,
            "phys_scale_pc": pix_scale_pc,
            "angular_size_arcmin": (shape[0] * pix_scale_deg * 60, shape[1] * pix_scale_deg * 60),
            "physical_size_pc": (shape[0] * pix_scale_pc, shape[1] * pix_scale_pc),
            "total_pixels": shape[0] * shape[1]
        }

    @staticmethod
    def compute_structure_function(Q: np.ndarray, U: np.ndarray, R_list: np.ndarray, 
                                   min_pI: Optional[float] = None, bin_width: float = 0.5) -> Dict[str, np.ndarray]:
        """
        Computes the polarization angle structure function D_phi(R) = <sin^2(phi_1 - phi_2)>.
        
        Args:
            Q (np.ndarray): Stokes Q map.
            U (np.ndarray): Stokes U map.
            R_list (np.ndarray): Array of pixel lags (radii) to compute the function over.
            min_pI (Optional[float]): Minimum polarized intensity threshold. Defaults to None.
            bin_width (float): Tolerance window for pixel pair distance matching. Defaults to 0.5.
            
        Returns:
            Dict[str, np.ndarray]: Dictionary with keys 'R' (radii), 'Dphi' (structure function), 'Npairs' (counts).
        """
        Q, U = np.asarray(Q, dtype=float), np.asarray(U, dtype=float)
        Ny, Nx = Q.shape
        pI = np.hypot(Q, U)
        
        if min_pI is None:
            min_pI = -np.inf
            
        valid = np.isfinite(Q) & np.isfinite(U) & (pI > min_pI)
        q = np.where(valid, Q / pI, np.nan)
        u = np.where(valid, U / pI, np.nan)

        Dphi = np.full_like(R_list, np.nan, dtype=float)
        Npairs = np.zeros_like(R_list, dtype=int)

        def accumulate_shift(dy: int, dx: int) -> Optional[np.ndarray]:
            y1s, y2s = slice(max(0, dy), min(Ny, Ny + dy)), slice(max(0, -dy), min(Ny, Ny - dy))
            x1s, x2s = slice(max(0, dx), min(Nx, Nx + dx)), slice(max(0, -dx), min(Nx, Nx - dx))
            q1, u1 = q[y1s, x1s], u[y1s, x1s]
            q2, u2 = q[y2s, x2s], u[y2s, x2s]
            m = np.isfinite(q1) & np.isfinite(q2)
            if not np.any(m): return None
            dq, du = q1[m] - q2[m], u1[m] - u2[m]
            return 0.25 * (dq**2 + du**2)

        for i, R in enumerate(R_list):
            if R < 0: continue
            rmax = int(np.ceil(R + bin_width))
            shifts = [(dy, dx) for dy in range(-rmax, rmax + 1) for dx in range(-rmax, rmax + 1) 
                      if (dx != 0 or dy != 0) and abs(np.hypot(dx, dy) - R) <= bin_width]

            contrib_list = [accumulate_shift(dy, dx) for dy, dx in shifts]
            valid_contribs = [c for c in contrib_list if c is not None and c.size > 0]
            if valid_contribs:
                contrib_vec = np.concatenate(valid_contribs)
                Dphi[i] = np.nanmean(contrib_vec)
                Npairs[i] = contrib_vec.size

        return {"R": R_list, "Dphi": Dphi, "Npairs": Npairs}

    def plot_polarization_map(self, I: np.ndarray, vx: np.ndarray, vy: np.ndarray, wcs: WCS, object_name: str, step: int = 4, save_path: str = None):
        """
        Plots the classic dust intensity map overlaid with magnetic field vectors.
        
        Args:
            I (np.ndarray): Stokes I map (background).
            vx (np.ndarray): B-field vector X component.
            vy (np.ndarray): B-field vector Y component.
            wcs (WCS): World Coordinate System for projection.
            object_name (str): Target name for the title.
            step (int): Decimation factor for the vector grid.
        """
        interval = ZScaleInterval()
        vmin, vmax = interval.get_limits(I)
        
        x = np.linspace(0, vx.shape[1] - 1, vx.shape[1])
        y = np.linspace(0, vx.shape[0] - 1, vx.shape[0])
        px, py = np.meshgrid(x, y)

        jcmtmap_mask = np.ones(I.shape)
        #jcmtmap_mask[I < vmax * 0.6] = np.nan

        fig = plt.figure(figsize=(7, 6))
        ax = fig.add_subplot(111, projection=wcs)
        im = ax.imshow(I, origin="lower", vmin=vmin, vmax=vmax, cmap='viridis')
        
        ax.quiver(
            px[::step, ::step] * jcmtmap_mask[::step, ::step],
            py[::step, ::step] * jcmtmap_mask[::step, ::step],
            vx[::step, ::step] * jcmtmap_mask[::step, ::step],
            vy[::step, ::step] * jcmtmap_mask[::step, ::step],
            angles='xy', scale_units='xy', scale=0.2,
            headwidth=0, headlength=0, headaxislength=0,
            pivot="mid", width=0.003, color="k"
        )

        plt.colorbar(im, ax=ax, pad=0.02, label=r"Stokes $I$")
        ax.set_xlabel("RA (J2000)")
        ax.set_ylabel("Dec (J2000)")
        ax.set_title(rf"{object_name} -- Magnetic Field Morphology")
        if save_path:
            fig.savefig(save_path, bbox_inches='tight')
            plt.close(fig)
        else:
            plt.show()
        
    def plot_snr_colored_polarization_map(self, I: np.ndarray, snr_map: np.ndarray, 
                                          vx: np.ndarray, vy: np.ndarray, wcs: WCS, 
                                          object_name: str, step: int = 4, save_path: str = None):
        """
        Plots a B-field map where vectors are colored by their SNR levels:
        1 < SNR <= 3 (cyan), 3 < SNR <= 5 (red), SNR > 5 (magenta).
        """
        vmin, vmax = self._get_robust_limits(I)
        
        x = np.linspace(0, vx.shape[1] - 1, vx.shape[1])
        y = np.linspace(0, vx.shape[0] - 1, vx.shape[0])
        px, py = np.meshgrid(x, y)

        intensity_mask = np.ones(I.shape)
        #intensity_mask[I < vmax * 0.6] = np.nan

        fig = plt.figure(figsize=(7, 6))
        ax = fig.add_subplot(111, projection=wcs)
        im = ax.imshow(I, origin="lower", vmin=vmin, vmax=vmax, cmap='viridis')

        # Цвета: бирюзовый вместо оранжевого
        levels = [1, 3, 5]
        colors = ['magenta', 'red', 'cyan']
        # Понятные математические подписи
        labels = [r'1 $<$ SNR $\le$ 3', r'3 $<$ SNR $\le$ 5', r'SNR $>$ 5']

        for i in range(len(levels)):
            lower = levels[i]
            upper = levels[i+1] if i+1 < len(levels) else np.inf
            
            snr_mask = np.where((snr_map > lower) & (snr_map <= upper), 1.0, np.nan)
            combined_mask = intensity_mask * snr_mask
            
            ax.quiver(
                px[::step, ::step] * combined_mask[::step, ::step],
                py[::step, ::step] * combined_mask[::step, ::step],
                vx[::step, ::step] * combined_mask[::step, ::step],
                vy[::step, ::step] * combined_mask[::step, ::step],
                angles='xy', scale_units='xy', scale=0.2,
                headwidth=0, headlength=0, headaxislength=0,
                pivot="mid", width=0.004, color=colors[i]
            )

        # Легенда с использованием прокси-артистов
        legend_elements = [Line2D([0], [0], color=c, lw=2, label=l) for c, l in zip(colors, labels)]
        ax.legend(handles=legend_elements, loc='upper right', fontsize=9, framealpha=0.8, edgecolor='black')

        plt.colorbar(im, ax=ax, pad=0.02, label=r"Stokes $I$")
        ax.set_xlabel("RA (J2000)")
        ax.set_ylabel("Dec (J2000)")
        ax.set_title(rf"{object_name.replace('_', ' ')} -- SNR-Coded B-Field")
        
        if save_path:
            fig.savefig(save_path, bbox_inches='tight')
            plt.close(fig)
        else:
            plt.show()
            
    def plot_structure_function(self, sf_data: Dict[str, np.ndarray], save_path: str = None):
            """
            Plots the raw and binned polarization angle structure function.
            """
            R, D, N = sf_data["R_pc"], sf_data["Dphi"], sf_data["Npairs"]
            m = np.isfinite(D) & (N > 0) & (R > 0)
            Rv, Dv, Nv = R[m], D[m], N[m]

            #m2 = np.isfinite(Dv) & (Dv > 0)
            #Rv, Dv, Nv = Rv[m2], Dv[m2], Nv[m2]
            
            if len(Rv) == 0:
                print(f"[Warning] No valid data for structure function plot. Skipping...")
                return
            
            sig = Dv / np.sqrt(Nv)

            def logbin_median(x, y, nbins=12):
                lx = np.log10(x)
                edges = np.linspace(lx.min(), lx.max(), nbins + 1)
                xb, yb = [], []
                for a, b in zip(edges[:-1], edges[1:]):
                    sel = (lx >= a) & (lx < b)
                    if np.any(sel):
                        xb.append(10**np.median(lx[sel]))
                        yb.append(np.median(y[sel]))
                return np.array(xb), np.array(yb)

            Rb_s, Db_s = logbin_median(Rv, Dv, nbins=10)

            fig, ax = plt.subplots(figsize=(8, 5))
            ax.plot(Rv, Dv, marker="o", ms=4, lw=1, alpha=0.6, color='gray', label=r"Raw $D_\phi(R)$")
            ax.errorbar(Rv, Dv, yerr=sig, fmt="none", ecolor='gray', lw=0.8, alpha=0.5)

            if len(Rb_s) > 1:
                ax.plot(Rb_s, Db_s, marker="s", ms=6, lw=2, color='black', label=r"Log-binned Median")

            ax.set_xscale("log")
            ax.set_yscale("log")
            ax.set_xlabel(r"Separation $R$ (pc)")
            ax.set_ylabel(r"Structure Function $D_\phi(R)$ (rad$^2$)")
            ax.set_title(r"Polarization Angle Dispersion")

            ax.minorticks_on()
            ax.grid(True, which="major", lw=0.8, alpha=0.3)
            ax.grid(True, which="minor", lw=0.4, alpha=0.15)
            ax.legend(frameon=True, edgecolor='black')

            if save_path:
                fig.savefig(save_path, bbox_inches='tight')
                plt.close(fig)
            else:
                plt.show()
            
            
    def plot_snr_distributions(self, p_frac: np.ndarray, psi_deg: np.ndarray, snr_pI: np.ndarray, object_name: str, save_path: Optional[str] = None):
        """
        Plots histograms of Polarization Fraction and EVPA for different 
        SNR thresholds (>1, >3, >5). Essential for analyzing Ricean bias 
        and the statistical ordering of the magnetic field.
        """
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
        
        thresholds = [1, 3, 5]
        colors = ['gray', 'royalblue', 'crimson']
        alphas = [0.4, 0.7, 0.9]
        
        for thr, col, alp in zip(thresholds, colors, alphas):
            mask = (snr_pI > thr) & np.isfinite(p_frac) & np.isfinite(psi_deg)
            valid_p = p_frac[mask]
            valid_psi = psi_deg[mask]
            
            if len(valid_p) == 0:
                continue
                
            # 1. Polarization Fraction Distribution
            bins_p = np.linspace(0, 0.3, 40) # Capped at 30% for visibility
            ax1.hist(valid_p, bins=bins_p, histtype='step', lw=2, 
                     color=col, alpha=alp, density=True, 
                     label=rf'$\sigma > {thr}$ ($N={len(valid_p)}$)')
            
            # 2. EVPA Distribution
            bins_psi = np.linspace(-90, 90, 36)
            ax2.hist(valid_psi, bins=bins_psi, histtype='step', lw=2, 
                     color=col, alpha=alp, density=True, 
                     label=rf'$\sigma > {thr}$')

        ax1.set_xlabel(r"Polarization Fraction $p$")
        ax1.set_ylabel("Probability Density")
        ax1.legend(frameon=False)
        ax1.grid(True, which="major", alpha=0.3)

        ax2.set_xlabel(r"EVPA $\psi$ (deg)")
        ax2.set_ylabel("Probability Density")
        ax2.legend(frameon=False)
        ax2.grid(True, which="major", alpha=0.3)

        fig.suptitle(rf"{object_name.replace('_', ' ')} -- Distributions by Polarized SNR")
        fig.tight_layout()
        
        if save_path:
            for fmt in ['pdf', 'png']:
                fig.savefig(f"{save_path}.{fmt}", bbox_inches='tight')
                plt.close(fig)
        else:
            plt.show()


    def detect_hills_statistically(self, I, wcs, object_name, margin_ratio=0.15, sigma_level=2.0, save_path=None):
        """
        Статистическая сегментация: выделяет только пиксели, значительно 
        превышающие средний фон внутри безопасной зоны.
        """

        # --- 1. ГЕОМЕТРИЯ (Ваш "забор") ---
        footprint = np.isfinite(I) & (I != 0)
        dist_map = distance_transform_edt(footprint)
        threshold_dist = np.nanmax(dist_map) * margin_ratio
        safe_zone_mask = dist_map >= threshold_dist

        # --- 2. СТАТИСТИКА ВНУТРИ ЗОНЫ ---
        # Выделяем значения Стокса I только внутри красного кольца
        data_in_zone = I[safe_zone_mask & np.isfinite(I)]
        
        if data_in_zone.size == 0:
            print(f"[WARN] Safe zone is empty for {object_name}")
            return

        mean_val = np.nanmean(data_in_zone)
        median_val = np.nanmedian(data_in_zone)
        std_val = np.nanstd(data_in_zone)
        # Порог: среднее + N сигм (подберите sigma_level, например 1.5 или 2.0)
        science_threshold = median_val + sigma_level * std_val #+median_val
        #print("sigma_level ",sigma_level)
        #print("std_val ", std_val)
        #print("median_val ", median_val)
        #print("mean_val ", mean_val)

        # --- 3. СОЗДАНИЕ МАСКИ ОБЛАКОВ ---
        # Оставляем только те пиксели, которые ОДНОВРЕМЕННО:
        # а) внутри safe_zone_mask
        # б) ярче порога (science_threshold)
        cloud_mask = (I > science_threshold) & safe_zone_mask
        cloud_mask_uint8 = cloud_mask.astype('uint8') * 255

        # Очистка: убираем мелкую "пыль" (объекты меньше 3x3 пикселя)
        #kernel = np.ones((2,2), np.uint8)
        #cloud_mask_cleaned = cv2.morphologyEx(cloud_mask_uint8, cv2.MORPH_OPEN, kernel, iterations=1)

        # Поиск контуров
        contours, _ = cv2.findContours(cloud_mask_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        # --- 4. ОТРИСОВКА ---
        fig = plt.figure(figsize=(8, 7))
        ax = fig.add_subplot(111, projection=wcs)
        
        # Используем Asinh для фона, чтобы видеть структуру
        norm = ImageNormalize(I, interval=ZScaleInterval(), stretch=AsinhStretch(a=0.1))
        im = ax.imshow(I, origin='lower', cmap='viridis', norm=norm)

        # Рисуем "забор" (красный)
        ax.contour(safe_zone_mask, levels=[0.5], colors='magenta', linewidths=1.2, alpha=0.8)
        
        # Рисуем "холмы" (голубые)
        count = 0
        for cnt in contours:
            if cv2.contourArea(cnt) > 10: # Фильтр мелкого мусора
                poly = cnt.reshape(-1, 2)
                poly = np.vstack([poly, poly[0]])
                ax.plot(poly[:, 0], poly[:, 1], color='red', lw=1.5)
                count += 1

        ax.set_title(fr"{object_name}: {count} clumps ($>${sigma_level}$\sigma$ above median)")
        plt.colorbar(im, ax=ax, pad=0.02, label=r"Stokes $I$")

        if save_path:
            plt.savefig(str(save_path), bbox_inches='tight')
            plt.close(fig)
        else:
            plt.show()
            
        return contours
            
    def process_object(self, object_name: str, distance_pc: float, band: str = "850um", step: int = 2):
        from scipy.ndimage import distance_transform_edt
        from astropy.visualization import ZScaleInterval
        
        print(f"\n{'='*60}\nPROCESSING: {object_name}\n{'='*60}")
        
        try:
            I, Q, U, wcs = self.load_stokes_maps(object_name, band)
        except FileNotFoundError:
            return None, None, None

        # --- ПОДГОТОВКА ПУТЕЙ ---
        report_dir = self.base_dir.parents[1] / "report"
        diag_dir = report_dir / "diagnostics" / object_name
        plots_dir = report_dir / "plots" / object_name
        for d in [diag_dir, plots_dir]: d.mkdir(parents=True, exist_ok=True)

        # --- ЭТАП 1: ГЛУБОКАЯ ДИАГНОСТИКА СТРУКТУР (OpenCV) ---
        # Это то, что мы только что исправили. Построит "забор" и выделит ядра.
        diag_path = diag_dir / f"{object_name}_stat_segmentation.pdf"
        self.detect_hills_statistically(I, wcs, object_name, 
                                 margin_ratio=0.15, 
                                 sigma_level=.5, 
                                 save_path=str(diag_path))
        # --- ЭТАП 2: ТЕХНИЧЕСКАЯ ОБРЕЗКА КРАЕВ (Distance Transform) ---
        pI_raw = np.hypot(Q, U)
        noise_pI_pre = self.estimate_rms(pI_raw)
        snr_pI_pre = pI_raw / noise_pI_pre if noise_pI_pre > 0 else np.zeros_like(I)
        
        footprint = np.isfinite(I) & (I != 0)
        dist_map = distance_transform_edt(footprint)
        threshold_dist = np.nanmax(dist_map) * 0.15 
        
        # "Ампутация" только зашумленных краев (для науки)
        bad_pixels = (snr_pI_pre > 3) & (dist_map < threshold_dist)
        I[bad_pixels], Q[bad_pixels], U[bad_pixels] = np.nan, np.nan, np.nan

        # --- ЭТАП 3: НАУЧНЫЙ АНАЛИЗ ЧИСТЫХ ДАННЫХ ---
        noise_I = self.estimate_rms(I)
        pI = np.hypot(Q, U)
        noise_pI = self.estimate_rms(pI)
        
        psi_deg = self.compute_evpa(Q, U)
        vx, vy = self.pol_vector_components(p=1.0, psi_rad=np.radians(psi_deg))
        scales = self.get_physical_scales(wcs, distance_pc, I.shape)

        # Стадии маскирования для отчета
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        axes[0].imshow(snr_pI_pre, origin='lower', vmin=0, vmax=10); axes[0].set_title("Initial SNR")
        axes[1].imshow(dist_map, origin='lower'); axes[1].set_title("Distance map")
        axes[2].imshow(~bad_pixels, origin='lower', cmap='gray'); axes[2].set_title("Masked Science Area")
        for ax in axes: ax.axis('off')
        fig.savefig(diag_dir / f"{object_name}_mask_stages.pdf", bbox_inches='tight')
        plt.close(fig)

        # --- ЭТАП 4: ФИНАЛЬНЫЕ ГРАФИКИ ---
        R_list_px = np.arange(1, 50)
        sf_data = self.compute_structure_function(Q, U, R_list=R_list_px, min_pI=3*noise_pI)
        sf_data["R_pc"] = R_list_px * scales["phys_scale_pc"]

        self.plot_polarization_map(I, vx, vy, wcs, object_name, step, save_path=plots_dir/f"{object_name}_pol_map.pdf")
        self.plot_structure_function(sf_data, save_path=plots_dir/f"{object_name}_sf.pdf")

        # Статистика
        valid_count = np.sum(~np.isnan(I))
        stats = {
            "Distance (pc)": f"{scales['distance_pc']}",
            "Effective Pixels": f"{valid_count}",
            "Stokes I RMS": f"{noise_I:.5f}",
            "Median SNR": f"{np.nanmedian(I/noise_I):.2f}"
        }

        return psi_deg, sf_data, stats    
    
    def process_object(self, object_name: str, distance_pc: float, band: str = "850um", step: int = 2):
        import cv2
        import numpy as np
        from astropy.visualization import ZScaleInterval
        
        print(f"\n{'='*60}\nPROCESSING: {object_name}\n{'='*60}")
        
        try:
            I, Q, U, wcs = self.load_stokes_maps(object_name, band)
        except FileNotFoundError:
            return None, None, None

        # --- ПОДГОТОВКА ПУТЕЙ ---
        report_dir = self.base_dir.parents[1] / "report"
        plots_dir = report_dir / "plots" / object_name
        plots_dir.mkdir(parents=True, exist_ok=True)

        # --- ЭТАП 1: ВЫДЕЛЕНИЕ КОНТУРОВ ОБЛАКОВ ---
        # График сегментации теперь сразу идет в целевую папку plots
        diag_path = plots_dir / f"{object_name}_stat_segmentation.pdf"
        
        contours = self.detect_hills_statistically(I, wcs, object_name, 
                                                   margin_ratio=0.15, 
                                                   sigma_level=0.5, 
                                                   save_path=str(diag_path))

        if not contours:
            print(f"[ERROR] No contours found for {object_name}. Check sigma_level.")
            return None, None, None

        # --- ЭТАП 2: ФИЗИЧЕСКОЕ МАСКИРОВАНИЕ (Ампутация фона) ---
        # 2.1 Создаем пустую маску (черный холст)
        science_mask = np.zeros(I.shape, dtype=np.uint8)
        
        # 2.2 Заливаем найденные контуры белым цветом (игнорируя мелкий мусор)
        for cnt in contours:
            if cv2.contourArea(cnt) > 10:
                cv2.drawContours(science_mask, [cnt], -1, 1, thickness=cv2.FILLED)
                
        science_mask_bool = science_mask.astype(bool)
        
        # 2.3 Уничтожаем все данные вне облаков. 
        # Теперь для numpy и scipy фона не существует.
        I[~science_mask_bool] = np.nan
        Q[~science_mask_bool] = np.nan
        U[~science_mask_bool] = np.nan

        # --- ЭТАП 3: НАУЧНЫЙ АНАЛИЗ (Только внутри контуров) ---
        # Шумы считаются только по выжившему газу
        noise_I = self.estimate_rms(I)
        pI = np.hypot(Q, U)
        noise_pI = self.estimate_rms(pI)
        
        # Вектора поляризации (автоматически станут NaN там, где Q и U равны NaN)
        psi_deg = self.compute_evpa(Q, U)
        vx, vy = self.pol_vector_components(p=1.0, psi_rad=np.radians(psi_deg))
        scales = self.get_physical_scales(wcs, distance_pc, I.shape)

        # --- ЭТАП 4: ФИНАЛЬНЫЕ ГРАФИКИ И СТАТИСТИКА ---
        R_list_px = np.arange(1, 50)
        
        # Структурная функция считается только по валидным пикселям
        sf_data = self.compute_structure_function(Q, U, R_list=R_list_px, min_pI=3*noise_pI)
        if sf_data is not None:
            sf_data["R_pc"] = R_list_px * scales["phys_scale_pc"]
            self.plot_structure_function(sf_data, save_path=plots_dir/f"{object_name}_sf.pdf")

        # Отрисовка карты с векторами (quiver автоматически проигнорирует NaN)
        self.plot_polarization_map(I, vx, vy, wcs, object_name, step, 
                                   save_path=plots_dir/f"{object_name}_pol_map.pdf")
        
        # 1. Убеждаемся, что карта SNR для поляризации рассчитана (если её еще нет)
        snr_pI_map = pI / noise_pI if noise_pI > 0 else np.zeros_like(pI)

        # 2. Определяем путь для сохранения
        snr_colored_path = plots_dir / f"{object_name}_snr_colored.pdf"

        # 3. Вызываем функцию
        self.plot_snr_colored_polarization_map(
            I=I, 
            snr_map=snr_pI_map, 
            vx=vx, 
            vy=vy, 
            wcs=wcs, 
            object_name=object_name, 
            step=step, 
            save_path=str(snr_colored_path)
        )

        # Суровая статистика
        valid_count = np.sum(science_mask_bool)
        stats = {
            "Distance (pc)": f"{scales['distance_pc']}",
            "Effective Pixels (in clouds)": f"{valid_count}",
            "Stokes I RMS": f"{noise_I:.5f}",
            "Pol. Int. RMS": f"{noise_pI:.5f}",
            "Median SNR (Clouds)": f"{np.nanmedian(I/noise_I):.2f}" if noise_I > 0 else "0"
        }

        return psi_deg, sf_data, stats
    
    
    
    def process_object(self, object_name: str, distance_pc: float, band: str = "850um", step: int = 2):
        """
        Центральный пайплайн анализа. 
        Изолирует структуры облака, применяет маску к Стоксовым параметрам 
        и генерирует полный комплект сравнительной графики.
        """
        import cv2
        import numpy as np
        from astropy.visualization import ZScaleInterval
        
        print(f"\n{'='*60}\nPROCESSING: {object_name}\n{'='*60}")
        
        try:
            I, Q, U, wcs = self.load_stokes_maps(object_name, band)
        except FileNotFoundError:
            return None, None, None

        # --- ПОДГОТОВКА ПУТЕЙ ---
        report_dir = self.base_dir.parents[1] / "report"
        plots_dir = report_dir / "plots" / object_name
        plots_dir.mkdir(parents=True, exist_ok=True)

        # --- ЭТАП 1: СЕГМЕНТАЦИЯ СТРУКТУР ---
        # Сразу сохраняем карту с контурами (тот самый "график как есть")
        diag_path = plots_dir / f"{object_name}_stat_segmentation.pdf"
        
        contours = self.detect_hills_statistically(I, wcs, object_name, 
                                                   margin_ratio=0.08, # Расширенная зона
                                                   sigma_level=0.5,   # Высокая чувствительность
                                                   save_path=str(diag_path))

        if not contours:
            print(f"[ERROR] No clumps found for {object_name}. Physics is canceled.")
            return None, None, None

        # --- ЭТАП 2: ФИЗИЧЕСКАЯ ИЗОЛЯЦИЯ ---
        science_mask = np.zeros(I.shape, dtype=np.uint8)
        
        # Заливаем контуры, создавая бинарный монолит
        for cnt in contours:
            if cv2.contourArea(cnt) > 5:
                cv2.drawContours(science_mask, [cnt], -1, 1, thickness=cv2.FILLED)
                
        science_mask_bool = science_mask.astype(bool)
        
        # Сохраняем оригинал для эстетики (подложка фоновых карт)
        I_full = I.copy()

        # Уничтожаем данные вне облаков. Математика здесь бессильна (и это хорошо).
        I[~science_mask_bool] = np.nan
        Q[~science_mask_bool] = np.nan
        U[~science_mask_bool] = np.nan

        # --- ЭТАП 3: РАСЧЕТЫ (Строго внутри маски) ---
        noise_I = self.estimate_rms(I)
        pI = np.hypot(Q, U)
        noise_pI = self.estimate_rms(pI)
        
        # Векторы обречены быть NaN за пределами контуров
        psi_deg = self.compute_evpa(Q, U)
        vx, vy = self.pol_vector_components(p=1.0, psi_rad=np.radians(psi_deg))
        scales = self.get_physical_scales(wcs, distance_pc, I_full.shape)

        snr_pI_map = pI / noise_pI if noise_pI > 0 else np.zeros_like(pI)

        # --- ЭТАП 4: ГРАФИКА И ВЫВОД ---
        R_list_px = np.arange(1, 50)
        sf_data = self.compute_structure_function(Q, U, R_list=R_list_px, min_pI=3*noise_pI)
        
        if sf_data is not None:
            sf_data["R_pc"] = R_list_px * scales["phys_scale_pc"]
            self.plot_structure_function(sf_data, save_path=plots_dir/f"{object_name}_sf.pdf")

        # 1. Одноцветные векторы на полном фоне
        self.plot_polarization_map(I_full, vx, vy, wcs, f"{object_name} (Full Context)", step, 
                                   save_path=plots_dir/f"{object_name}_pol_map_full.pdf")

        # 2. Одноцветные векторы только на ядрах
        self.plot_polarization_map(I, vx, vy, wcs, f"{object_name} (Isolated Cores)", step, 
                                   save_path=plots_dir/f"{object_name}_pol_map_cropped.pdf")

        # 3. SNR-векторы на полном фоне
        self.plot_snr_colored_polarization_map(I_full, snr_pI_map, vx, vy, wcs, f"{object_name} (Full Context)", step, 
                                               save_path=plots_dir/f"{object_name}_snr_colored_full.pdf")

        # 4. SNR-векторы только на ядрах
        self.plot_snr_colored_polarization_map(I, snr_pI_map, vx, vy, wcs, f"{object_name} (Isolated Cores)", step, 
                                               save_path=plots_dir/f"{object_name}_snr_colored_cropped.pdf")

        # Статистика
        valid_count = np.sum(science_mask_bool)
        stats = {
            "Distance (pc)": f"{scales['distance_pc']}",
            "Effective Pixels": f"{valid_count}",
            "Stokes I RMS": f"{noise_I:.5f}",
            "Median SNR": f"{np.nanmedian(I/noise_I):.2f}" if noise_I > 0 else "0"
        }

        return psi_deg, sf_data, stats

# Implemetation for one objrct

In [79]:
CLEAN_ARCHIVE_PATH = "data/BISTRO_Clean"
analyzer = PolarizationAnalyzer(base_archive_path=CLEAN_ARCHIVE_PATH)

object_name = "Auriga"
object_dist = obj_dist_step_pc[object_name][0]
step = obj_dist_step_pc[object_name][1]

evpa_map, sf_results, stats = analyzer.process_object(
    object_name=object_name, 
    distance_pc=object_dist, 
    band="850um",
    step = step
)


PROCESSING: Auriga


# Implementation for all objs in dir with report in .tex

In [81]:
import os
from pathlib import Path

# Инициализация анализатора
CLEAN_ARCHIVE_PATH = Path("data/BISTRO_Clean")
analyzer = PolarizationAnalyzer(base_archive_path=CLEAN_ARCHIVE_PATH)

# ====================================================================
# 1. СТАТИЧЕСКАЯ ПРЕАМБУЛА И МЕТОДОЛОГИЯ
# ====================================================================
# Преамбула описывает переход от сырых данных к изолированным физическим ядрам.
latex_content = r"""\documentclass[10pt, a4paper]{article}
\usepackage[margin=1.5cm]{geometry}
\usepackage{graphicx}
\usepackage{booktabs}
\usepackage{subcaption}
\usepackage{float}
\usepackage{amsmath}
\usepackage{mathpazo}
\usepackage[utf8]{inputenc}
\usepackage[T2A]{fontenc}
\usepackage[english,russian]{babel}

\title{Polarimetric Analysis of Dust Continuum in Molecular Clouds}
\author{BISTRO Data Pipeline}
\date{\today}

\begin{document}

\maketitle

\section*{Methodology and Processing Parameters}

This report presents the automated polarimetric analysis of dust continuum emission. The pipeline abandons arbitrary geometric cropping in favor of rigorous statistical isolation of dense molecular clumps.

\paragraph{Statistical Segmentation and Masking:}
To mitigate edge artifacts, a geometric safety margin is established via distance transform. Within this zone, physical structures are isolated using a dynamic threshold based on the local median and standard deviation of Stokes $I$ ($I > \tilde{I} + N\sigma_I$). Morphological dilation ensures the preservation of compact cores and the continuity of diffuse filaments. The background is strictly masked (assigned \texttt{NaN}), ensuring that polarization parameters and subsequent statistics are calculated exclusively within the identified physical structures.

\paragraph{Plot Structure:}
For each targeted region, the analysis yields three diagnostic panels:
\begin{enumerate}
    \item \textbf{Statistical Segmentation (Top Left):} Isophotes and morphological boundaries of the isolated dense cores over the asinh-stretched Stokes $I$ continuum.
    \item \textbf{SNR-Coded Magnetic Field (Top Right):} Polarization vectors ($E$-field rotated by 90$^\circ$) overlaid on the full continuum map. Vectors are strictly constrained to the segmented areas and color-coded by the polarized intensity signal-to-noise ratio ($\text{SNR}_{pI}$).
    \item \textbf{Structure Function (Bottom Center):} Angular dispersion of the magnetic field $D_\phi(R)$ plotted against the spatial lag $R$ (in parsecs), calculated strictly within the isolated clumps.
\end{enumerate}

\newpage
"""

# ====================================================================
# 2. ГЕНЕРАЦИЯ СТРАНИЦ ОБЪЕКТОВ
# ====================================================================

for object_name, params in obj_dist_step_pc.items():
    object_dist = params[0]
    step = params[1]
    ref = params[2]
    
    # Запуск пайплайна
    result = analyzer.process_object(
        object_name=object_name, 
        distance_pc=object_dist, 
        band="850um",
        step=step
    )
    
    if result[0] is None:
        continue  # Пропуск объектов без валидных данных
        
    psi_deg, sf_data, stats = result
    safe_name = object_name.replace('_', r'\_')
    
    # Формирование строк таблицы статистики
    table_rows = ""
    for key, value in stats.items():
        safe_key = key.replace('_', r'\_')
        safe_value = str(value).replace('_', r'\_')
        table_rows += f"        {safe_key} & {safe_value} \\\\\n"
        
    table_rows += f"        Distance Reference & {ref} \\\\\n"
    table_rows += f"        Vector Decimation Step & {step} px \\\\\n"

    # Сборка страницы объекта: 2 графика сверху (карты), 1 снизу по центру (функция)
    latex_content += f"""\\section*{{Target: {safe_name}}}

\\begin{{table}}[H]
    \\centering
    \\renewcommand{{\\arraystretch}}{{1.1}}
    \\begin{{tabular}}{{ll}}
        \\toprule
        \\textbf{{Parameter}} & \\textbf{{Value}} \\\\
        \\midrule
{table_rows}        \\bottomrule
    \\end{{tabular}}
\\end{{table}}

\\vspace{{0.5cm}}

\\begin{{figure}}[H]
    \\centering
    \\begin{{subfigure}}{{0.48\\textwidth}}
        \\includegraphics[width=\\linewidth]{{../plots/{object_name}/{object_name}_stat_segmentation.pdf}}
    \\end{{subfigure}}
    \\hfill
    \\begin{{subfigure}}{{0.48\\textwidth}}
        \\includegraphics[width=\\linewidth]{{../plots/{object_name}/{object_name}_snr_colored_full.pdf}}
    \\end{{subfigure}}
    
    \\vspace{{0.5cm}}
    
    \\begin{{subfigure}}{{0.6\\textwidth}}
        \\includegraphics[width=\\linewidth]{{../plots/{object_name}/{object_name}_sf.pdf}}
    \\end{{subfigure}}
\\end{{figure}}

\\newpage
"""

# ====================================================================
# 3. ЗАВЕРШЕНИЕ И СОХРАНЕНИЕ
# ====================================================================
latex_content += r"\end{document}"

# Формирование путей
report_dir = CLEAN_ARCHIVE_PATH.parents[1] / "report"
text_dir = report_dir / "report_text"
text_dir.mkdir(parents=True, exist_ok=True)

report_path = text_dir / "automated_bistro_report.tex"
with open(report_path, "w", encoding="utf-8") as f:
    f.write(latex_content)

print(f"\n[OK] Полный научный LaTeX-отчет сгенерирован: {report_path.resolve()}")


PROCESSING: Auriga

PROCESSING: L1495

PROCESSING: L43_Karoly_2023

PROCESSING: MonR2

PROCESSING: NGC1333
[Warning] Shape mismatch for NGC1333. Synchronizing Q/U to I: (205, 158)

PROCESSING: OMC1_450

PROCESSING: OMC1_850

PROCESSING: Perseus_B1

PROCESSING: Rosette

PROCESSING: l1689b

PROCESSING: oph_a

PROCESSING: oph_b

PROCESSING: oph_c

[OK] Полный научный LaTeX-отчет сгенерирован: /Users/ildana/astro/nu/mach_number/report/report_text/automated_bistro_report.tex
